# Soft Decision Trees

**Paper:** İrsoy, O., Yıldız, O. T., & Alpaydın, E. (2012). *Soft Decision Trees*. ICPR 2012.

Standard decision trees make **hard** splits — a sample goes left or right based on a threshold. 
Soft Decision Trees make **soft** splits — every sample reaches every leaf with some probability.
This makes the tree fully differentiable and trainable end-to-end with backpropagation.

At each internal node $i$:
$$p_i(\mathbf{x}) = \sigma(\mathbf{w}_i^\top \mathbf{x} + b_i)$$

Final prediction is a weighted sum over leaf distributions:
$$P(y \mid \mathbf{x}) = \sum_\ell \mu_\ell(\mathbf{x}) \cdot Q_\ell(y)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import load_iris, make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

from neural_trees import SoftDecisionTree

plt.style.use('seaborn-v0_8-whitegrid')
print('All imports OK')

## 1. Basic usage on Iris

In [ ]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

sdt = SoftDecisionTree(depth=4, max_epochs=40, learning_rate=0.01, verbose=True)
sdt.fit(X_train_s, y_train)

print(f"\nTest accuracy: {sdt.score(X_test_s, y_test):.4f}")

## 2. Training curve

In [ ]:
history = sdt.training_history_
epochs = [h['epoch'] for h in history]
losses = [h['loss'] for h in history]
accs   = [h['accuracy'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, losses, color='#667eea', linewidth=2)
ax1.set_title('Training Loss', fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')

ax2.plot(epochs, accs, color='#48bb78', linewidth=2)
ax2.set_title('Training Accuracy', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 3. What each leaf learned

Each leaf stores a class distribution. This tells us which leaf specializes in which class.

In [ ]:
leaf_dists = sdt.get_leaf_distributions()  # (n_leaves, n_classes)
class_names = ['setosa', 'versicolor', 'virginica']

fig, ax = plt.subplots(figsize=(14, 4))
x = np.arange(len(leaf_dists))
width = 0.25
colors = ['#667eea', '#f093fb', '#48bb78']

for i, (cls, color) in enumerate(zip(class_names, colors)):
    ax.bar(x + i * width, leaf_dists[:, i], width, label=cls, color=color, alpha=0.85)

ax.set_xlabel('Leaf index')
ax.set_ylabel('Class probability')
ax.set_title('Class distributions stored in each leaf node', fontsize=13)
ax.set_xticks(x + width)
ax.set_xticklabels([f'Leaf {i}' for i in range(len(leaf_dists))], rotation=45)
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print("Leaves with strong class specialization (p > 0.8):")
for i, dist in enumerate(leaf_dists):
    dominant = dist.max()
    if dominant > 0.8:
        cls = class_names[dist.argmax()]
        print(f"  Leaf {i:2d} → {cls} ({dominant:.2f})")

## 4. Decision boundary — SDT vs hard Decision Tree

Using 2D moons dataset so we can visualize the boundary.

In [ ]:
def plot_boundary(ax, clf, X, y, scaler, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                          np.linspace(y_min, y_max, 300))
    grid = scaler.transform(np.c_[xx.ravel(), yy.ravel()])
    Z = clf.predict(grid).reshape(xx.shape)
    # Convert string labels to int if needed
    try:
        Z = Z.astype(int)
    except ValueError:
        from sklearn.preprocessing import LabelEncoder
        Z = LabelEncoder().fit_transform(Z.ravel()).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='white', linewidth=0.5, s=40)
    ax.set_title(title, fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])

X2, y2 = make_moons(n_samples=400, noise=0.2, random_state=42)
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.2, random_state=42)
sc2 = StandardScaler()
X2_tr_s = sc2.fit_transform(X2_tr)
X2_te_s = sc2.transform(X2_te)

# Soft Decision Tree
sdt2 = SoftDecisionTree(depth=4, max_epochs=50)
sdt2.fit(X2_tr_s, y2_tr)
sdt_acc = sdt2.score(X2_te_s, y2_te)

# Hard Decision Tree
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X2_tr_s, y2_tr)
dt_acc = dt.score(X2_te_s, y2_te)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_boundary(axes[0], dt,   X2_tr, y2_tr, sc2, f'Hard Decision Tree (depth=4)\nTest acc: {dt_acc:.3f}')
plot_boundary(axes[1], sdt2, X2_tr, y2_tr, sc2, f'Soft Decision Tree (depth=4)\nTest acc: {sdt_acc:.3f}')
plt.suptitle('Hard vs Soft Decision Boundary — Moons Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'Hard DT  test accuracy: {dt_acc:.4f}')
print(f'Soft DT  test accuracy: {sdt_acc:.4f}')

## 5. Effect of depth on accuracy

In [ ]:
depths = [2, 3, 4, 5, 6]
sdt_accs, dt_accs = [], []

for d in depths:
    s = SoftDecisionTree(depth=d, max_epochs=30)
    s.fit(X2_tr_s, y2_tr)
    sdt_accs.append(s.score(X2_te_s, y2_te))
    
    h = DecisionTreeClassifier(max_depth=d, random_state=42)
    h.fit(X2_tr_s, y2_tr)
    dt_accs.append(h.score(X2_te_s, y2_te))

plt.figure(figsize=(8, 5))
plt.plot(depths, sdt_accs, 'o-', color='#667eea', linewidth=2, markersize=8, label='Soft DT')
plt.plot(depths, dt_accs,  's--', color='#f093fb', linewidth=2, markersize=8, label='Hard DT')
plt.xlabel('Tree depth')
plt.ylabel('Test accuracy')
plt.title('Accuracy vs Depth: Soft DT vs Hard DT', fontsize=13)
plt.legend()
plt.ylim(0.7, 1.0)
plt.tight_layout()
plt.show()

## 6. sklearn Pipeline compatibility

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('sdt', SoftDecisionTree(max_epochs=20)),
])

param_grid = {
    'sdt__depth': [3, 4, 5],
    'sdt__penalty_coef': [1e-4, 1e-3],
}

gs = GridSearchCV(pipe, param_grid, cv=3, n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)

print(f'Best params:    {gs.best_params_}')
print(f'Best CV score:  {gs.best_score_:.4f}')
print(f'Test accuracy:  {gs.score(X_test, y_test):.4f}')